In [11]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.linear_model import LogisticRegression,SGDClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import MinMaxScaler

current_dir = Path.cwd()
project_root = current_dir.parents[2]

full_set_path_HY3 = project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION/HY3/FULL_SET/'
full_set_path_HY4 = project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION/HY4/FULL_SET/'
full_set_path_MCID = project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION/MCID/FULL_SET/'

feature_selection_path_HY3 = project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION/HY3/FEATURE_SELECTION/'
feature_selection_path_HY4 = project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION/HY4/FEATURE_SELECTION/'
feature_selection_path_MCID = project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION/MCID/FEATURE_SELECTION/'


classification_models = {
    "decision_tree": DecisionTreeClassifier(random_state=42),

    "random_forest": RandomForestClassifier(
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"

    ),

    "extra_trees": ExtraTreesClassifier(
        random_state=42,
        n_jobs=-1
    ),

    "xgboost": XGBClassifier(
    tree_method="hist",
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42
    ),

    "adaboost": AdaBoostClassifier(
        algorithm="SAMME",
        random_state=42
    ),

    "svm": SVC(
        kernel="rbf",
        probability=True,
        random_state=42
    ),

    "logistic_regression": LogisticRegression(
        random_state=42,
        n_jobs=-1,
        max_iter=10000,
        class_weight="balanced"
    ),

    "knn": KNeighborsClassifier(
        n_jobs=-1
    ),

    "gaussian_nb": GaussianNB()

    
}

In [12]:
import json

with open(project_root/"SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/Domain_data.json", "r") as archivo:
    datos = json.load(archivo)

print(datos)


{'M_data': ['NP1COG_mean', 'NP1COG_min', 'NP1COG_max', 'NP1COG_var', 'NP1COG_std', 'NP1HALL_mean', 'NP1HALL_min', 'NP1HALL_max', 'NP1HALL_var', 'NP1HALL_std', 'NP1DPRS_mean', 'NP1DPRS_min', 'NP1DPRS_max', 'NP1DPRS_var', 'NP1DPRS_std', 'NP1ANXS_mean', 'NP1ANXS_min', 'NP1ANXS_max', 'NP1ANXS_var', 'NP1ANXS_std', 'NP1APAT_mean', 'NP1APAT_min', 'NP1APAT_max', 'NP1APAT_var', 'NP1APAT_std', 'NP1DDS_mean', 'NP1DDS_min', 'NP1DDS_max', 'NP1DDS_var', 'NP1DDS_std', 'NP1SLPN_mean', 'NP1SLPN_min', 'NP1SLPN_max', 'NP1SLPN_var', 'NP1SLPN_std', 'NP1SLPD_mean', 'NP1SLPD_min', 'NP1SLPD_max', 'NP1SLPD_var', 'NP1SLPD_std', 'NP1PAIN_mean', 'NP1PAIN_min', 'NP1PAIN_max', 'NP1PAIN_var', 'NP1PAIN_std', 'NP1URIN_mean', 'NP1URIN_min', 'NP1URIN_max', 'NP1URIN_var', 'NP1URIN_std', 'NP1CNST_mean', 'NP1CNST_min', 'NP1CNST_max', 'NP1CNST_var', 'NP1CNST_std', 'NP1LTHD_mean', 'NP1LTHD_min', 'NP1LTHD_max', 'NP1LTHD_var', 'NP1LTHD_std', 'NP1FATG_mean', 'NP1FATG_min', 'NP1FATG_max', 'NP1FATG_var', 'NP1FATG_std', 'NP2SPCH_m

# BAYESIAN OPTIMIZATION
- FEATURE SELECTION: chi2_mi_union_topk
- HYPERPARAMETER TURNING

In [41]:
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import mutual_info_classif, chi2, VarianceThreshold
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical



def evaluate_models_nested_bayes(
    X_df: pd.DataFrame,
    y_df: pd.DataFrame,
    models: dict,
    outer_splits: int = 5,
    inner_splits: int = 10,
    test_size: float = 0.3,
    random_state: int = 42,
    decimals: int = 4,
    n_iter_search: int = 40,
    n_jobs_search: int = -1,
):
    y = y_df.iloc[:, 0].to_numpy()
    X = X_df.to_numpy()
    classes = np.unique(y)

    def build_pipeline(estimator):
        return Pipeline([
            ("scaler", MinMaxScaler()),
            ("model", clone(estimator)),
        ])

    def get_search_spaces(model_name):
        spaces = {
            "decision_tree": {
                "model__max_depth": Integer(2, 30),
                "model__min_samples_split": Integer(2, 20),
                "model__min_samples_leaf": Integer(1, 10),
                "model__criterion": Categorical(["gini", "entropy"]),
                "model__max_features": Categorical([None, "sqrt"]),
                "model__class_weight": Categorical([
                    None,
                    "balanced"]),
            },

            "random_forest": {
                "model__n_estimators": Integer(200, 800),
                "model__max_depth": Integer(4, 30),
                "model__min_samples_split": Integer(2, 20),
                "model__min_samples_leaf": Integer(1, 10),
                "model__max_features": Categorical(["sqrt", "log2"]),
                "model__bootstrap": Categorical([True]),
                "model__class_weight": Categorical([
                    None,
                    "balanced",
                    "balanced_subsample"]),
            },

            "extra_trees": {
                "model__n_estimators": Integer(200, 800),
                "model__max_depth": Integer(4, 30),
                "model__min_samples_split": Integer(2, 20),
                "model__min_samples_leaf": Integer(1, 10),
                "model__max_features": Categorical(["sqrt", "log2"]),
                "model__class_weight": Categorical([
                    None,
                    "balanced",
                    "balanced_subsample"]),
            },

            "xgboost": {
                "model__n_estimators": Integer(200, 600),
                "model__max_depth": Integer(3, 10),
                "model__learning_rate": Real(0.01, 0.3, prior="log-uniform"),
                "model__subsample": Real(0.6, 1.0),
                "model__colsample_bytree": Real(0.6, 1.0),
                "model__min_child_weight": Integer(1, 10),
                "model__gamma": Real(1e-8, 5.0, prior="log-uniform"),
                "model__reg_alpha": Real(1e-8, 5.0, prior="log-uniform"),
                "model__reg_lambda": Real(1e-8, 5.0, prior="log-uniform"),
            },

            "adaboost": {
                "model__n_estimators": Integer(50, 500),
                "model__learning_rate": Real(0.01, 1.0, prior="log-uniform"),
            },

            "svm": {
                "model__C": Real(1e-3, 1e3, prior="log-uniform"),
                "model__gamma": Real(1e-5, 1.0, prior="log-uniform"),
                "model__kernel": Categorical(["rbf"]),
                "model__class_weight": Categorical([
                    None,
                    "balanced"]),
            },

            "logistic_regression": {
                "model__C": Real(1e-4, 1e2, prior="log-uniform"),
                "model__solver": Categorical(["lbfgs", "saga"]),
                "model__penalty": Categorical(["l2"]),
                "model__class_weight": Categorical([
                    None,
                    "balanced"]),
            },

            "knn": {
                "model__n_neighbors": Integer(3, 51),
                "model__weights": Categorical(["uniform", "distance"]),
                "model__p": Integer(1, 2),
            },

            "gaussian_nb": {
                "model__var_smoothing": Real(1e-10, 1e-6, prior="log-uniform"),
            }}
        
        if model_name not in spaces:
            raise ValueError(f"No search space definido para {model_name}")

        return spaces[model_name]

    def compute_metrics(y_true, y_pred, y_proba):
        return {
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "Recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "F1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "AUC_macro": roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro"),
            "Precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
            "Recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
            "F1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
            "AUC_weighted": roc_auc_score(y_true, y_proba, multi_class="ovr", average="weighted"),
        }

    def summarize(metrics_list, suffix):
        df = pd.DataFrame(metrics_list)
        mean = df.mean(numeric_only=True)
        std = df.std(ddof=1, numeric_only=True)
        return {
            f"{col}_{suffix}": f"{mean[col]:.{decimals}f} ± {std[col]:.{decimals}f}"
            for col in df.columns
        }

    class TqdmBayesCallback:
        """
        Callback para actualizar una barra tqdm en cada iteración
        del BayesSearchCV.
        """
        def __init__(self, pbar):
            self.pbar = pbar

        def __call__(self, res):
            # En skopt, normalmente se minimiza la función objetivo.
            # Como scoring='f1_macro' se maximiza internamente usando el negativo.
            # Por eso recuperamos el mejor score como -min(func_vals).
            best_cv = -np.min(res.func_vals)

            self.pbar.update(1)
            self.pbar.set_postfix({
                "best_inner_cv_f1": f"{best_cv:.4f}"
            })

            return False  # continuar optimización

    outer = StratifiedShuffleSplit(
        n_splits=outer_splits,
        test_size=test_size,
        random_state=random_state
    )

    test_summary_rows = []
    cv_summary_rows = []
    best_params_rows = []

    # Recorremos modelo por modelo
    for model_idx, (model_name, estimator) in enumerate(models.items(), start=1):
        print(f"\nEvaluating {model_name} with Bayesian Search...")

        test_metrics_all = []
        cv_metrics_all = []
        best_params_per_outer_fold = []

        search_spaces = get_search_spaces(model_name)

        # Barra principal del modelo: outer CV
        outer_pbar = tqdm(
            total=outer_splits,
            desc=f"[{model_idx}/{len(models)}] {model_name} OUTER",
            position=0,
            leave=True
        )

        for fold_id, (train_idx, test_idx) in enumerate(outer.split(X, y), start=1):
            X_train, y_train = X[train_idx], y[train_idx]
            X_test, y_test = X[test_idx], y[test_idx]

            inner = StratifiedKFold(
                n_splits=inner_splits,
                shuffle=True,
                random_state=random_state
            )

            base_pipeline = build_pipeline(estimator)

            # Subbarra: optimización bayesiana del inner CV
            inner_pbar = tqdm(
                total=n_iter_search,
                desc=f"Outer fold {fold_id}/{outer_splits} | Bayes {n_iter_search} iters | inner_cv={inner_splits}",
                position=1,
                leave=False
            )

            opt = BayesSearchCV(
                estimator=base_pipeline,
                search_spaces=search_spaces,
                n_iter=n_iter_search,
                scoring="f1_macro",
                cv=inner,
                n_jobs=n_jobs_search,
                refit=True,
                random_state=random_state,
                verbose=0,
            )

            callback = TqdmBayesCallback(inner_pbar)
            opt.fit(X_train, y_train, callback=callback)

            inner_pbar.close()

            best_model = opt.best_estimator_
            best_params_per_outer_fold.append(opt.best_params_)

            cv_metrics_all.append({
                "F1_macro": opt.best_score_,
            })

            test_pred = best_model.predict(X_test)
            test_proba_raw = best_model.predict_proba(X_test)

            test_classes = best_model.named_steps["model"].classes_
            test_proba = np.zeros((len(y_test), len(classes)))

            for j, cls in enumerate(test_classes):
                test_proba[:, np.where(classes == cls)[0][0]] = test_proba_raw[:, j]

            outer_metrics = compute_metrics(y_test, test_pred, test_proba)
            test_metrics_all.append(outer_metrics)

            avg_outer_f1 = np.mean([m["F1_macro"] for m in test_metrics_all])
            avg_inner_cv_f1 = np.mean([m["F1_macro"] for m in cv_metrics_all])

            outer_pbar.update(1)
            outer_pbar.set_postfix({
                "fold": f"{fold_id}/{outer_splits}",
                "avg_outer_f1": f"{avg_outer_f1:.4f}",
                "avg_inner_cv_f1": f"{avg_inner_cv_f1:.4f}",
            })

        outer_pbar.close()

        test_summary_rows.append(
            pd.Series(summarize(test_metrics_all, "Testing"), name=model_name)
        )

        cv_summary_rows.append(
            pd.Series(summarize(cv_metrics_all, "CV"), name=model_name)
        )

        best_params_rows.append(
            pd.Series({"Best_Params_Outer_Folds": best_params_per_outer_fold}, name=model_name)
        )

    df_test_summary = pd.DataFrame(test_summary_rows)[[
        "Accuracy_Testing",
        "Precision_macro_Testing",
        "Recall_macro_Testing",
        "F1_macro_Testing",
        "AUC_macro_Testing",
        "Precision_weighted_Testing",
        "Recall_weighted_Testing",
        "F1_weighted_Testing",
        "AUC_weighted_Testing"
    ]]

    df_cv_summary = pd.DataFrame(cv_summary_rows)[[
        "F1_macro_CV"
    ]]

    df_best_params = pd.DataFrame(best_params_rows)

    df_final_summary = pd.concat(
        [df_test_summary, df_cv_summary, df_best_params],
        axis=1
    )

    return df_final_summary

## SET FULL DATA



In [42]:
X_HY3_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY3_full.csv", index_col=0)
y_HY3_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY3_full.csv", index_col=0)
print("X_HY3_data shape:", X_HY3_data.shape)
print("y_HY3_data shape:", y_HY3_data.shape)
X_HY3_data.head()

X_HY3_data shape: (909, 931)
y_HY3_data shape: (909, 1)


,ENRLLRRK2,ENRLGBA,SEX,RAWHITE,EDUCYRS,AGE_AT_VISIT,MCAALTTM_mean,MCACUBE_mean,MCACLCKC_mean,MCACLCKN_mean,...,NP3PTRMR_std,NP3PTRML_std,NP3KTRMR_std,NP3KTRML_std,NP3RTARU_std,NP3RTALU_std,NP3RTARL_std,NP3RTALL_std,NP3RTALJ_std,NP3RTCON_std
PATNO,,,,,,,,,,,,,,,,,,,,,
3003,0,0,0.0,1.0,16.0,59.7,1.000000,1.000000,1.0,1.0,...,0.577350,0.577350,0.000000,0.57735,1.154701,0.000000,0.000000,0.0,0.0,0.577350
3018,0,0,0.0,1.0,16.0,63.6,1.000000,0.666667,1.0,1.0,...,1.527525,1.154701,1.154701,0.57735,0.577350,0.577350,0.577350,0.0,0.0,1.527525
3020,0,0,0.0,1.0,15.0,77.0,0.666667,0.666667,1.0,1.0,...,1.154701,0.000000,0.000000,0.00000,0.577350,1.154701,1.154701,0.0,0.0,1.154701
3028,0,0,1.0,1.0,22.0,78.8,1.000000,1.000000,1.0,1.0,...,0.000000,0.000000,0.000000,0.57735,0.000000,1.527525,0.000000,0.0,0.0,1.527525
3051,0,0,1.0,1.0,18.0,74.7,1.000000,1.000000,1.0,1.0,...,0.577350,0.000000,0.000000,0.00000,0.577350,0.000000,0.000000,0.0,0.0,0.577350


In [43]:
df_HY3_MOTOR_SELECTOR_bayesian = evaluate_models_nested_bayes(
    X_df=X_HY3_data,
    y_df=y_HY3_data,
    models=classification_models
)

df_HY3_MOTOR_SELECTOR_bayesian.to_csv(
    full_set_path_HY3 / "HY3_MOTOR_bayesian.csv",
    index=False
)

df_HY3_MOTOR_SELECTOR_bayesian.head(10)


Evaluating decision_tree with Bayesian Search...


[1/9] decision_tree OUTER:   0%|          | 0/5 [00:00<?, ?it/s]

Outer fold 1/5 | Bayes 40 iters | inner_cv=10:   0%|          | 0/40 [00:00<?, ?it/s]

/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/numpy/ma/core.py:2885: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,
/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/numpy/ma/core.py:2885: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,
/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/numpy/ma/core.py:2885: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Outer fold 2/5 | Bayes 40 iters | inner_cv=10:   0%|          | 0/40 [00:00<?, ?it/s]

...

Outer fold 1/5 | Bayes 40 iters | inner_cv=10:   0%|          | 0/40 [00:00<?, ?it/s]

Outer fold 2/5 | Bayes 40 iters | inner_cv=10:   0%|          | 0/40 [00:00<?, ?it/s]

Outer fold 3/5 | Bayes 40 iters | inner_cv=10:   0%|          | 0/40 [00:00<?, ?it/s]

Outer fold 4/5 | Bayes 40 iters | inner_cv=10:   0%|          | 0/40 [00:00<?, ?it/s]

KeyboardInterrupt: 